In [3]:
import os
import ssl
import pg8000
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine
from supabase import create_client, Client

# 1. Load các biến môi trường từ file .env ở thư mục gốc của dự án
# Tìm .env từ vị trí của file test này (hoặc từ current working directory nếu chạy trong notebook)
try:
    ROOT_DIR = Path(__file__).resolve().parents[1]
except NameError:
    ROOT_DIR = Path.cwd().parent
env_path = ROOT_DIR / ".env"
load_dotenv(env_path)

# Lấy thông tin cấu hình từ file .env
DB_HOST = os.getenv("DB_HOST")
DB_PORT = int(os.getenv("DB_PORT", 5432))
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

if not all([DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD]):
    raise ValueError("Thiếu thông tin cấu hình cơ sở dữ liệu trong file .env")
# Tạo Connection URI sử dụng driver pg8000
db_uri = f"postgresql+pg8000://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# Khởi tạo SQLAlchemy Engine với SSL cấu hình qua connect_args
ssl_context = ssl.create_default_context()
ssl_context.check_hostname = False
ssl_context.verify_mode = ssl.CERT_NONE
engine = create_engine(db_uri, connect_args={"ssl_context": ssl_context})

In [4]:
query = """
        SELECT
            fse.gen_id, fse.site_id, fse.geo_id, fse.date_id, fse.time_id,
            fse.energy_generated_kwh,
            ds.campus_name, dg.location_name, dg.latitude, dg.longitude,
            dd.full_date, dd.month, dd.year,
            dt.time_string, dt.hour,
            ds.capacity_kw AS site_capacity
        FROM datawarehouse.fact_solar_energy_gen fse
        JOIN datawarehouse.dim_solar_site ds ON fse.site_id = ds.site_id
        JOIN datawarehouse.dim_geography dg ON fse.geo_id = dg.geo_id
        JOIN datawarehouse.dim_date dd ON fse.date_id = dd.date_id
        JOIN datawarehouse.dim_time dt ON fse.time_id = dt.time_id
        LIMIT 10000;
        """

df_result = pd.read_sql_query(query, engine)
print(df_result)

      gen_id  site_id  geo_id   date_id  time_id  energy_generated_kwh  \
0      40321       27      27  20200210      215                0.0000   
1      40322       41      41  20200210      215                0.0000   
2      40323       42      42  20200210      215                0.0000   
3      40324        1       1  20200210      230                0.0000   
4      40325        2       2  20200210      230                0.0000   
...      ...      ...     ...       ...      ...                   ...   
9995   50316        9       9  20200219     1330                1.8610   
9996   50317       10      10  20200219     1330                7.6100   
9997   50318       12      12  20200219     1330               15.8125   
9998   50319       13      13  20200219     1330                7.8125   
9999   50320       27      27  20200219     1330               77.4218   

         campus_name   location_name   latitude   longitude   full_date  \
0           Bundoora        Bundoora

In [5]:
processing_path = ROOT_DIR / "data" / "processed"
output_name = "fact_solar_energy_gen.csv"
if not processing_path.exists():
    processing_path.mkdir(parents=True, exist_ok=True)

df_result.to_csv(processing_path / output_name, index=False)